# Roomscape: Colab reconstruction

Run the Roomscape pipeline on Google Colab (extract frames -> COLMAP -> Gaussian splat -> collision -> export).
Use a GPU runtime (Runtime > Change runtime type > T4/L4/A100).

In [ ]:
!nvidia-smi
!apt-get -qq install -y colmap ffmpeg  # apt COLMAP is CPU-only

In [ ]:
!pip -q install nerfstudio
!pip -q install -r requirements.txt

In [ ]:
from google.colab import files
import os

up = files.upload()  # upload video (.mp4/.mov)
video_file = list(up.keys())[0]
print(f"Uploaded: {video_file}")

In [ ]:
# Video sampling parameters
FRAMES = 300
TRIM_START = 0.0
TRIM_END = 0.0

In [ ]:
import yaml

with open("pipeline/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["colmap"]["use_gpu"] = False

with open("pipeline/config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Configured colmap.use_gpu: false")

In [ ]:
flags = []
if FRAMES:
    flags.append(f"--frames {FRAMES}")
if TRIM_START:
    flags.append(f"--trim-start {TRIM_START}")
if TRIM_END:
    flags.append(f"--trim-end {TRIM_END}")
flags_str = " ".join(flags)

!python pipeline/run_pipeline.py --video {video_file} --out out {flags_str}

In [ ]:
import glob
from google.colab import files

outputs = glob.glob("out/scene/*") + glob.glob("out/splat.ply") + glob.glob("out/report.*")
for f in outputs:
    files.download(f)